# EXP-020 프롬프트 v6(H22)의 원인 분석

EXP-020(프롬프트 v4 → v6, 기준·후보 모두 j_v12)의 원인 분석이다. 판정은 보류(종합 지표 +0.011, 관련성 안전 조건 실패)이고 사용자가 예외 채택하지 않았다(D-67). 새 API 호출은 없다. 질의·답변 원문은 출력하지 않고 수만 낸다.

In [1]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd

from rag.gen_eval.common import read_jsonl


In [2]:
ROOT = Path.cwd().parent
BASE_JUDGE = ROOT / "data/gen_eval/judge/hybrid-cond-filter-law-ms256-k20-ce-v4-dev-full-spk-head-deepseek_go_mixed-j_v12.jsonl"  # 기준(EXP-019 생성 기록, j_v12 재채점)
CAND_JUDGE = ROOT / "data/gen_eval/judge/hybrid-cond-filter-law-ms256-k20-ce-v6-dev-full-spk-head-deepseek_go_mixed-j_v12.jsonl"  # 후보(EXP-020)
BASE_RUNS = ROOT / "data/gen_eval/runs/hybrid-cond-filter-law-ms256-k20-ce-v4-dev-full-spk-head.jsonl"
CAND_RUNS = ROOT / "data/gen_eval/runs/hybrid-cond-filter-law-ms256-k20-ce-v6-dev-full-spk-head.jsonl"
MISMATCH = "발언자 불일치"
# 질의자 주장을 사실처럼 쓴 진술을 0 판정 이유에서 찾는 어림(notebooks/38_01과 같은 패턴)
CLAIM = re.compile(r"(질의자|위원)")
CLAIM_KIND = re.compile(r"(확인|사실처럼|단정|주장)")
CITE = re.compile(r"\[(\d+)\]")


In [3]:
def by_qid(path: Path) -> dict:
    return {r["qid"]: r for r in read_jsonl(path)}


def speaker_names(run: dict) -> set[str]:
    """답변이 인용한 근거의 머리 줄 화자 이름 중 답변에 나온 것."""
    cited = {int(n) for n in CITE.findall(run["answer"])}
    names = {p["name"] for c in run["contexts"] if c["n"] in cited and c.get("speakers") for f in ("questioners", "answerers") for p in c["speakers"][f]}
    return {n for n in names if n and n in run["answer"]}


def zero_kind(v: dict) -> str:
    """0 판정 이유의 종류. 낱말 대조 어림이다."""
    if MISMATCH in v["reason"]:
        return "발언자 불일치"
    if CLAIM.search(v["reason"]) and CLAIM_KIND.search(v["reason"]):
        return "질의자 주장"
    return "그 밖"


## 데이터 로딩


In [4]:
base, cand = by_qid(BASE_JUDGE), by_qid(CAND_JUDGE)
base_runs, cand_runs = by_qid(BASE_RUNS), by_qid(CAND_RUNS)
print(len(base), len(cand), len(base_runs), len(cand_runs), set(base) == set(cand))


330 330 330 330 True


In [5]:
rows = []
for q in base:
    b, c = base[q], cand[q]
    rows.append({
        "qid": q,
        "f_base": (b.get("faithfulness") or {}).get("score"),
        "f_cand": (c.get("faithfulness") or {}).get("score"),
        "r_base": (b.get("relevance") or {}).get("score"),
        "r_cand": (c.get("relevance") or {}).get("score"),
        "named": bool(speaker_names(cand_runs[q])),
        "named_base": bool(speaker_names(base_runs[q])),
        "chars_base": len(base_runs[q]["answer"]),
        "chars_cand": len(cand_runs[q]["answer"]),
    })
df = pd.DataFrame(rows).set_index("qid")
df["f_diff"] = df.f_cand - df.f_base
df["r_diff"] = df.r_cand - df.r_base
print(df.shape, df.named.sum())
df[["f_diff", "r_diff"]].describe().round(3)


(330, 10) 308


,f_diff,r_diff
count,291.000,298.000
mean,0.045,-0.020
std,0.244,0.252
min,-1.000,-1.000
25%,0.000,0.000
50%,0.000,0.000
75%,0.000,0.000
max,1.000,1.000


## 충실성: 화자 이름이 나온 답변과 나오지 않은 답변

후보 답변이 인용 근거의 화자 이름을 적었는지로 나눠 충실성 차이의 몫을 본다. 기여는 그 묶음 차이의 합을 두 실행에서 계산된 전체 질의 수로 나눈 값이다.


In [6]:
f = df.dropna(subset=["f_diff"])
parts = f.groupby("named").f_diff.agg(질의="count", 평균차이="mean", 합="sum", 상승=lambda s: (s > 0).sum(), 하락=lambda s: (s < 0).sum())
parts["기여"] = parts["합"] / len(f)
print(f"전체 {len(f)}질의, 평균 차이 {f.f_diff.mean():+.3f}")
parts.round(3)


전체 291질의, 평균 차이 +0.045


,질의,평균차이,합,상승,하락,기여
named,,,,,,
False,2,0.000,0.000,0,0,0.000
True,289,0.045,13.013,50,30,0.045


In [7]:
def zero_table(judged: dict) -> Counter:
    c = Counter()
    for r in judged.values():
        for v in (r.get("faithfulness") or {}).get("verdicts") or []:
            c["진술"] += 1
            if v["verdict"] == 0:
                c["0 판정"] += 1
                c[zero_kind(v)] += 1
    return c


pd.DataFrame({"기준": zero_table(base), "후보": zero_table(cand)}).fillna(0).astype(int)


,기준,후보
진술,746,868
0 판정,104,70
그 밖,22,12
질의자 주장,63,2
발언자 불일치,19,56


In [8]:
def named_statements(run: dict, r: dict) -> tuple[int, int, int]:
    """(이름이 든 진술, 그중 1, 이름이 없는 진술 중 1의 비율 계산용 수)."""
    names = speaker_names(run)
    vs = (r.get("faithfulness") or {}).get("verdicts") or []
    hit = [v for v in vs if any(n in v["statement"] for n in names)]
    return len(hit), sum(v["verdict"] for v in hit), len(vs)


tot = [named_statements(cand_runs[q], cand[q]) for q in cand]
hit, ok, all_st = map(sum, zip(*tot, strict=True))
all_ok = sum(v["verdict"] for r in cand.values() for v in (r.get("faithfulness") or {}).get("verdicts") or [])
print(f"후보 진술 {all_st}개 중 이름이 든 진술 {hit}개, 그중 1 판정 {ok}개({ok / hit:.3f})")
print(f"이름이 없는 진술 {all_st - hit}개 중 1 판정 {all_ok - ok}개({(all_ok - ok) / (all_st - hit):.3f})")


후보 진술 868개 중 이름이 든 진술 836개, 그중 1 판정 771개(0.922)
이름이 없는 진술 32개 중 1 판정 27개(0.844)


## 답변 관련성: 하락 질의

관련성 하락 질의를 화자 이름 유무, 라벨 변화, 답변 길이 변화로 나눈다.


In [9]:
r = df.dropna(subset=["r_diff"])
parts = r.groupby("named").r_diff.agg(질의="count", 평균차이="mean", 합="sum", 상승=lambda s: (s > 0).sum(), 하락=lambda s: (s < 0).sum())
parts["기여"] = parts["합"] / len(r)
print(f"전체 {len(r)}질의, 평균 차이 {r.r_diff.mean():+.3f}")
parts.round(3)


전체 298질의, 평균 차이 -0.020


,질의,평균차이,합,상승,하락,기여
named,,,,,,
False,2,0.00,0.0,0,0,0.00
True,296,-0.02,-6.0,20,32,-0.02


In [10]:
moves = Counter((base[q]["relevance"]["label"], cand[q]["relevance"]["label"]) for q in r.index if r.r_diff[q] != 0)
pd.Series(moves).sort_values(ascending=False)


답함      일부        26
일부      답함        12
답함      답하지 않음     4
답하지 않음  답함         4
        일부         4
일부      답하지 않음     2
dtype: int64

In [11]:
ratio = (r.chars_cand / r.chars_base)
print("답변 길이 비(후보/기준) 중앙값:", round(ratio.median(), 2))
print(ratio.groupby(r.r_diff.pipe(lambda s: s.gt(0).map({True: "상승"}).fillna(s.lt(0).map({True: "하락"})).fillna("같음"))).median().round(2).to_string())
print("하락 질의 중 이름 답변:", int(((r.r_diff < 0) & r.named).sum()), "/", int((r.r_diff < 0).sum()))
print("전체 질의 중 이름 답변 비율:", round(r.named.mean(), 3))


답변 길이 비(후보/기준) 중앙값: 1.17
r_diff
같음    1.15
상승    1.52
하락    1.19
하락 질의 중 이름 답변: 32 / 32
전체 질의 중 이름 답변 비율: 0.993


In [12]:
down = [q for q in r.index if r.r_diff[q] < 0]
print("하락 질의의 후보 충실성 변화 평균:", round(df.loc[down, "f_diff"].mean(), 3))
print("하락 질의의 후보 부분 거부 수:", sum(cand[q]["refusal"] == "partial" for q in down), "기준:", sum(base[q]["refusal"] == "partial" for q in down))


하락 질의의 후보 충실성 변화 평균: 0.203
하락 질의의 후보 부분 거부 수: 5 기준: 5


## 기준 답변의 이름 유무로 나누기

후보 답변은 거의 모두 이름 답변이라 위 분해로는 나뉘지 않는다. 기준 답변에 이름이 없던 질의(새로 이름이 나온 답변)와 있던 질의로 다시 나눈다.

In [13]:
for name, col in (("충실성", "f_diff"), ("관련성", "r_diff")):
    sub = df.dropna(subset=[col])
    parts = sub.groupby("named_base")[col].agg(질의="count", 평균차이="mean", 합="sum", 상승=lambda s: (s > 0).sum(), 하락=lambda s: (s < 0).sum())
    parts["기여"] = parts["합"] / len(sub)
    print(name, f"전체 {len(sub)}질의, 평균 차이 {sub[col].mean():+.3f}")
    print(parts.round(3).to_string())

충실성 전체 291질의, 평균 차이 +0.045
             질의   평균차이       합  상승  하락     기여
named_base                                   
False       151  0.095  14.343  37  11  0.049
True        140 -0.009  -1.330  13  19 -0.005
관련성 전체 298질의, 평균 차이 -0.020
             질의   평균차이    합  상승  하락     기여
named_base                                
False       153 -0.026 -4.0  11  19 -0.013
True        145 -0.014 -2.0   9  13 -0.007


## 판독 결과 (API 호출 없음)

답변·근거를 대조한 결과의 수만 적는다. 서브에이전트 판단 1회이고 메인이 일부를 확인했다. 표본은 seed 20260929로 뽑았다.

- 이름 답변 표본 30개: 잘못 귀속 7, 경계 1, 맞음 22. 유형은 질의자 발언을 답변자 말로 3, 답변자 발언을 질의자 말로 2, 다른 화자 2이고 judge가 0으로 잡은 것은 4다. 첫 문장에 답한 답변 17/30, 질의자 말만 전한 답변 5
- "발언자 불일치" 진술 표본 30개(전체 56개): judge 맞음 23, 오판 7. 맞은 23개의 유형은 질의자 발언을 답변자 말로 17, 답변자 발언을 질의자 말로 4, 다른 화자 2이고 라벨 구간 안 11, 밖 12다

이름 답변 308개에 표본 비율을 곱한 어림은 잘못 귀속 답변 약 70개(EXP-019 147개 × 4/30 ≈ 20개)다. 표본 30개라 구간이 넓다.